# Multi-Future Electron Density Prediction with Graph Attention (GAT)

This notebook combines the trajectory prediction of Notebook 2 with the attention-based message passing of Notebook 3.

### Key Features:
- **Trajectory Output**: Predicts density at $t+5, t+10$, and $t+15$ simultaneously.
- **Attention Mechanism**: Uses Multi-head GAT (Graph Attention Network) to weighted spatial and physical interactions.
- **Attention Inspection**: Tools to analyze which nodes are most critical for future forecasts.
- **Architecture**: Residual GAT with multi-head trajectory branches.

In [1]:
# 1) Move to repo (if needed)
# !git clone <your-repo-url>
# %cd /content/Predicting-Electron-Interactions-as-Evolving-Graphs/notebooks

# 2) Install dependencies
!pip -q install numpy pandas matplotlib seaborn scikit-learn tqdm scipy h5py

import torch, os
torch_ver = torch.__version__.split('+')[0]
cuda_tag = "cu121" if torch.cuda.is_available() else "cpu"
!pip -q install torch-geometric
!pip -q install pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html

print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.8 MB/s eta 0:00:00
Torch: 2.10.0+cpu CUDA: False


In [2]:
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv

np.random.seed(42)
torch.manual_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


In [3]:
# Data path detection logic
RAW_DATA_DIR = '../data/raw'

def resolve_raw_data_dir():
    candidates = [
        Path('../data/raw'),
        Path('./data/raw'),
        Path('/content/drive/MyDrive/Predicting-Electron-Interactions-as-Evolving-Graphs/data/raw'),
    ]
    for p in candidates:
        if (p / 'ammonia_x').exists():
            return p.resolve()
    return Path(RAW_DATA_DIR)

raw_data_dir = resolve_raw_data_dir()
ammonia_dir = raw_data_dir / 'ammonia_x'
print(f'Using data directory: {raw_data_dir}')

def load_density_file(filepath):
    # Predictions are normalized for stability later
    return np.loadtxt(filepath, usecols=1)

ammonia_files = sorted(glob.glob(str(ammonia_dir / 'rvlab.tdscf.rho.*')))
print(f'Ammonia files: {len(ammonia_files)}')

Using data directory: ../data/raw
Ammonia files: 0


In [4]:
def build_hybrid_edges(density, num_nodes, k_seq=5, k_knn=10):
    """Combines spatial (grid) and physical (density) proximity."""
    edge_index = []
    # Sequential (Grid)
    for i in range(num_nodes):
        for j in range(max(0, i - k_seq), min(num_nodes, i + k_seq + 1)):
            if i != j: edge_index.append([i, j])
    
    # KNN (Density)
    nbrs = NearestNeighbors(n_neighbors=k_knn + 1, algorithm='auto').fit(density.reshape(-1, 1))
    indices = nbrs.kneighbors(density.reshape(-1, 1), return_distance=False)
    for i, neighbors in enumerate(indices):
        for j in neighbors[1:]:
            edge_index.append([i, j])
            
    return torch.tensor(edge_index, dtype=torch.long).t().contiguous().unique(dim=1)

In [5]:
class MultiFutureDataset(Dataset):
    def __init__(self, files, lookback=1, future_offsets=[5, 10, 15], k_knn=5):
        super().__init__()
        self.files = files
        self.lookback = lookback
        self.offsets = future_offsets
        self.k = k_knn
        
        # Pre-load for normalization stats
        sample = load_density_file(files[0])
        self.num_nodes = len(sample)
        
    def len(self):
        return len(self.files) - self.lookback - max(self.offsets)

    def get(self, idx):
        x_raw = load_density_file(self.files[idx])
        targets = []
        for off in self.offsets:
            targets.append(load_density_file(self.files[idx + off]))
        
        # Normalize features (Crucial for GAT stability)
        mu, std = x_raw.mean(), x_raw.std() + 1e-6
        x_norm = (x_raw - mu) / std
        y_norm = [(t - mu) / std for t in targets]
        
        x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(np.column_stack(y_norm), dtype=torch.float32)
        
        edge_index = build_hybrid_edges(x_norm, self.num_nodes, k_knn=self.k)
        
        return Data(x=x, edge_index=edge_index, y=y)

dataset = MultiFutureDataset(ammonia_files[:100], k_knn=3) # Small subset for testing
print(f"Dataset ready. Size: {len(dataset)}")

IndexError: list index out of range

In [ ]:
class MultiFutureAttentionGNN(nn.Module):
    """
    Combining GATv2 for attention with multi-head future prediction.
    This model predicts the future so you don't have to. :)
    """
    def __init__(self, hidden_dim=128, heads=4, num_layers=4, num_futures=3):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden_dim)
        
        self.gat_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        for i in range(num_layers):
            self.gat_layers.append(GATv2Conv(hidden_dim, hidden_dim // heads, heads=heads))
            self.norms.append(nn.LayerNorm(hidden_dim))
            
        # Task-specific heads for each future timestamp
        self.future_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1)
            ) for _ in range(num_futures)
        ])

    def forward(self, data, return_attn=False):
        x, edge_index = data.x, data.edge_index
        x = self.input_proj(x)
        
        last_attn = None
        for gat, norm in zip(self.gat_layers, self.norms):
            h = x
            if return_attn:
                x, last_attn = gat(x, edge_index, return_attention_weights=True)
            else:
                x = gat(x, edge_index)
            x = norm(x + h) # Residual connection
            x = F.elu(x)
            
        outputs = [head(x).squeeze(-1) for head in self.future_heads]
        out = torch.stack(outputs, dim=-1)
        
        if return_attn:
            return out, last_attn
        return out

model = MultiFutureAttentionGNN(hidden_dim=64, heads=2, num_layers=3).to(device)
print(model)

In [ ]:
loader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print("Starting micro-training loop...")
model.train()
for epoch in range(10):
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.6f}")

In [ ]:
def inspect_attention(model, dataset_sample):
    model.eval()
    with torch.no_grad():
        pred, (attn_edge_index, attn_weights) = model(dataset_sample.to(device), return_attn=True)
    
    # Average across heads for visualization
    scores = attn_weights.mean(dim=1).cpu().numpy()
    
    # Get top 10 edges
    top_idx = np.argsort(scores)[-10:][::-1]
    
    print("Top Attended Edges (Latent Physical Interactions):")
    for i in top_idx:
        u, v = attn_edge_index[:, i].cpu().numpy()
        print(f"Node {u:5d} <--- Node {v:5d} | Score: {scores[i]:.4f}")

inspect_attention(model, dataset[0])

In [ ]:
def plot_comparison(model, dataset_sample):
    model.eval()
    with torch.no_grad():
        pred = model(dataset_sample.to(device)).cpu().numpy()
    target = dataset_sample.y.numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    times = [5, 10, 15]
    for i, t in enumerate(times):
        axes[i].scatter(target[:, i], pred[:, i], alpha=0.3, s=5, color='teal')
        axes[i].plot([target.min(), target.max()], [target.min(), target.max()], 'r--', lw=2)
        axes[i].set_title(f"Forecast at t+{t}")
        axes[i].set_xlabel("Ground Truth")
        axes[i].set_ylabel("Prediction")
        
    plt.suptitle("Multi-Future Attention GNN Performance", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_comparison(model, dataset[0])